# MiniGPT Playground

Interactive exploration of the GPT model we built from scratch.

In [1]:
import torch
import sys
sys.path.insert(0, '.')

from minigpt.config import GPT_CONFIG_124M
from minigpt.model import GPTModel
from minigpt.generate import generate
from minigpt.layernorm import LayerNorm
from minigpt.feedforward import GELU, FeedForward
from minigpt.attention import MultiHeadAttention
from minigpt.block import TransformerBlock

/Users/dondapatisushanth.reddy/Documents/personal/projects/minigpt/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 1. The Config

Every hyperparameter for GPT-2 124M:

In [2]:
for key, val in GPT_CONFIG_124M.items():
    print(f"{key:>20s} = {val}")

          vocab_size = 50257
      context_length = 1024
             emb_dim = 768
             n_heads = 12
            n_layers = 12
           drop_rate = 0.1
            qkv_bias = False


## 2. Build the Full Model

In [3]:
model = GPTModel(GPT_CONFIG_124M)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"That's ~{total_params / 1e6:.1f}M parameters")

Total parameters: 124,412,160
That's ~124.4M parameters


### Parameter Breakdown

Where do the 124M parameters live?

In [4]:
print(f"{'Component':<40s} {'Shape':<25s} {'Params':>12s}")
print("-" * 80)

for name, p in model.named_parameters():
    print(f"{name:<40s} {str(tuple(p.shape)):<25s} {p.numel():>12,}")

print("-" * 80)
print(f"{'TOTAL':<40s} {'':<25s} {total_params:>12,}")

# Note: tok_emb.weight and lm_head.weight are the SAME tensor (weight tying)
# so the count above doesn't double-count them
unique_params = sum(p.numel() for p in set(model.parameters()))
print(f"{'UNIQUE (weight-tied)':<40s} {'':<25s} {unique_params:>12,}")

Component                                Shape                           Params
--------------------------------------------------------------------------------
tok_emb.weight                           (50257, 768)                38,597,376
pos_emb.weight                           (1024, 768)                    786,432
blocks.0.ln1.scale                       (768,)                             768
blocks.0.ln1.shift                       (768,)                             768
blocks.0.attn.W_q.weight                 (768, 768)                     589,824
blocks.0.attn.W_k.weight                 (768, 768)                     589,824
blocks.0.attn.W_v.weight                 (768, 768)                     589,824
blocks.0.attn.W_o.weight                 (768, 768)                     589,824
blocks.0.attn.W_o.bias                   (768,)                             768
blocks.0.ln2.scale                       (768,)                             768
blocks.0.ln2.shift                     

## 3. Inspect Individual Components

### LayerNorm

In [5]:
ln = LayerNorm(emb_dim=4)
x = torch.tensor([[[2.0, 4.0, 6.0, 8.0]]])
out = ln(x)

print(f"Input:      {x[0, 0].tolist()}")
print(f"Mean:       {x[0, 0].mean().item():.2f}")
print(f"Var:        {x[0, 0].var(correction=0).item():.2f}")
print(f"Normalized: {out[0, 0].tolist()}")
print(f"Out mean:   {out[0, 0].mean().item():.6f} (should be ~0)")
print(f"Out var:    {out[0, 0].var(correction=0).item():.6f} (should be ~1)")

Input:      [2.0, 4.0, 6.0, 8.0]
Mean:       5.00
Var:        5.00
Normalized: [-1.341639518737793, -0.44721317291259766, 0.44721317291259766, 1.341639518737793]
Out mean:   0.000000 (should be ~0)
Out var:    0.999998 (should be ~1)


### GELU Activation

In [6]:
gelu = GELU()
test_values = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]
x = torch.tensor(test_values)
out = gelu(x)

print(f"{'Input':>8s} | {'GELU':>8s} | {'ReLU':>8s} | {'Difference':>10s}")
print("-" * 42)
for inp, g in zip(test_values, out.tolist()):
    relu = max(0, inp)
    print(f"{inp:>8.1f} | {g:>8.4f} | {relu:>8.4f} | {g - relu:>10.4f}")

   Input |     GELU |     ReLU | Difference
------------------------------------------
    -3.0 |  -0.0036 |   0.0000 |    -0.0036
    -2.0 |  -0.0454 |   0.0000 |    -0.0454
    -1.0 |  -0.1588 |   0.0000 |    -0.1588
    -0.5 |  -0.1543 |   0.0000 |    -0.1543
     0.0 |   0.0000 |   0.0000 |     0.0000
     0.5 |   0.3457 |   0.5000 |    -0.1543
     1.0 |   0.8412 |   1.0000 |    -0.1588
     2.0 |   1.9546 |   2.0000 |    -0.0454
     3.0 |   2.9964 |   3.0000 |    -0.0036


### Attention Weights Visualization

Look at how attention distributes across positions (with causal mask):

In [7]:
small_cfg = {
    "emb_dim": 16, "n_heads": 2, "context_length": 8,
    "qkv_bias": False, "drop_rate": 0.0,
}
mha = MultiHeadAttention(small_cfg)
mha.eval()

x = torch.randn(1, 5, 16)
q = mha.W_q(x).view(1, 5, 2, 8).transpose(1, 2)
k = mha.W_k(x).view(1, 5, 2, 8).transpose(1, 2)

scores = q @ k.transpose(-2, -1) / (8 ** 0.5)
scores = scores.masked_fill(mha.mask[:5, :5], float("-inf"))
weights = torch.softmax(scores, dim=-1)

print("Attention weights (Head 0):")
print("Rows = queries (each token), Cols = keys (what it attends to)")
print()
w = weights[0, 0].detach()
labels = [f"pos{i}" for i in range(5)]
print(f"{'':>6s}", *[f"{l:>7s}" for l in labels])
for i, row in enumerate(w):
    print(f"{labels[i]:>6s}", *[f"{v:>7.3f}" for v in row.tolist()])

Attention weights (Head 0):
Rows = queries (each token), Cols = keys (what it attends to)

          pos0    pos1    pos2    pos3    pos4
  pos0   1.000   0.000   0.000   0.000   0.000
  pos1   0.611   0.389   0.000   0.000   0.000
  pos2   0.462   0.259   0.279   0.000   0.000
  pos3   0.370   0.231   0.189   0.210   0.000
  pos4   0.219   0.177   0.187   0.190   0.227


## 4. Forward Pass — Trace the Shapes

Watch the data flow through the entire model:

In [8]:
token_ids = torch.tensor([[100, 200, 300, 400, 500]])
print(f"Input token IDs:     {token_ids.shape}  →  {token_ids[0].tolist()}")

tok_emb = model.tok_emb(token_ids)
print(f"Token embeddings:    {tok_emb.shape}")

pos_emb = model.pos_emb(torch.arange(5))
print(f"Position embeddings: {pos_emb.shape}")

x = tok_emb + pos_emb
print(f"Combined (tok+pos):  {x.shape}")

for i, block in enumerate(model.blocks):
    x = block(x)
    if i < 3 or i >= 10:
        print(f"After block {i:>2d}:       {x.shape}  mean={x.mean().item():>7.4f}  std={x.std().item():>7.4f}")
    elif i == 3:
        print(f"  ...")

x = model.final_norm(x)
print(f"After final norm:    {x.shape}  mean={x.mean().item():>7.4f}  std={x.std().item():>7.4f}")

logits = model.lm_head(x)
print(f"Logits:              {logits.shape}")
print(f"  (that's {logits.shape[1]} positions × {logits.shape[2]} vocab tokens)")

Input token IDs:     torch.Size([1, 5])  →  [100, 200, 300, 400, 500]
Token embeddings:    torch.Size([1, 5, 768])
Position embeddings: torch.Size([5, 768])
Combined (tok+pos):  torch.Size([1, 5, 768])
After block  0:       torch.Size([1, 5, 768])  mean=-0.0041  std= 1.4594
After block  1:       torch.Size([1, 5, 768])  mean=-0.0027  std= 1.4885
After block  2:       torch.Size([1, 5, 768])  mean=-0.0062  std= 1.5230
  ...
After block 10:       torch.Size([1, 5, 768])  mean=-0.0048  std= 1.7525
After block 11:       torch.Size([1, 5, 768])  mean= 0.0113  std= 1.7765
After final norm:    torch.Size([1, 5, 768])  mean= 0.0000  std= 1.0001
Logits:              torch.Size([1, 5, 50257])
  (that's 5 positions × 50257 vocab tokens)


## 5. Generate Text (Random Weights)

The model has random weights, so the output will be nonsense.
But it proves the full pipeline works: IDs in → model → greedy decoding → IDs out.

In [10]:
!pip install tiktoken

  Using cached regex-2026.9.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.4 MB/s  0:00:00 eta 0:00:01
Using cached regex-2026.9.3-cp313-cp313-macosx_11_0_arm64.whl (291 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [tiktoken]5/7 [requests]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [11]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False
    print("tiktoken not installed — using raw token IDs")
    print("Install with: pip install tiktoken")

In [17]:
if HAS_TIKTOKEN:
    prompt = "hey, hbgbhgb"
    prompt_ids = enc.encode(prompt)
    print(f"Prompt: {repr(prompt)}")
    print(f"Token IDs: {prompt_ids}")
    print(f"Tokens: {[enc.decode([t]) for t in prompt_ids]}")
else:
    prompt_ids = [15496, 11, 995]
    print(f"Using hardcoded IDs for 'Hello, world': {prompt_ids}")

input_tensor = torch.tensor([prompt_ids])
output = generate(model, input_tensor, max_new_tokens=9, context_length=1024)
generated_ids = output[0].tolist()

print(f"\nGenerated IDs: {generated_ids}")

if HAS_TIKTOKEN:
    print(f"Generated text: {repr(enc.decode(generated_ids))}")
    print(f"\n(Nonsense because weights are random — training comes next!)")

Prompt: 'hey, hbgbhgb'
Token IDs: [20342, 11, 289, 65, 22296, 71, 22296]
Tokens: ['hey', ',', ' h', 'b', 'gb', 'h', 'gb']

Generated IDs: [20342, 11, 289, 65, 22296, 71, 22296, 22296, 22296, 22296, 22296, 22296, 22296, 22296, 22296, 22296]
Generated text: 'hey, hbgbhgbgbgbgbgbgbgbgbgbgb'

(Nonsense because weights are random — training comes next!)


## 6. Weight Tying Proof

Verify that the token embedding and LM head share the same weight matrix:

In [18]:
print(f"tok_emb.weight shape: {model.tok_emb.weight.shape}")
print(f"lm_head.weight shape: {model.lm_head.weight.shape}")
print(f"Same object in memory: {model.tok_emb.weight is model.lm_head.weight}")
print(f"Data pointer matches:  {model.tok_emb.weight.data_ptr() == model.lm_head.weight.data_ptr()}")

# Modify one, the other changes too
original_val = model.tok_emb.weight[0, 0].item()
model.tok_emb.weight.data[0, 0] = 999.0
print(f"\nSet tok_emb.weight[0,0] = 999.0")
print(f"lm_head.weight[0,0] = {model.lm_head.weight[0, 0].item()} (same!)")
model.tok_emb.weight.data[0, 0] = original_val  # restore

tok_emb.weight shape: torch.Size([50257, 768])
lm_head.weight shape: torch.Size([50257, 768])
Same object in memory: True
Data pointer matches:  True

Set tok_emb.weight[0,0] = 999.0
lm_head.weight[0,0] = 999.0 (same!)


## 7. Causal Mask Proof

Changing future tokens must not affect past outputs:

In [19]:
ids = torch.tensor([[10, 20, 30, 40, 50]])
logits1 = model(ids)

ids_modified = ids.clone()
ids_modified[0, 3] = 99  # change token at position 3
ids_modified[0, 4] = 99  # change token at position 4
logits2 = model(ids_modified)

for pos in range(5):
    same = torch.allclose(logits1[0, pos], logits2[0, pos], atol=1e-5)
    status = "SAME" if same else "DIFFERENT"
    print(f"Position {pos}: {status}")

print("\nPositions 0-2 are SAME (can't see positions 3-4)")
print("Positions 3-4 are DIFFERENT (they changed, or see changed tokens)")

Position 0: SAME
Position 1: SAME
Position 2: SAME
Position 3: DIFFERENT
Position 4: DIFFERENT

Positions 0-2 are SAME (can't see positions 3-4)
Positions 3-4 are DIFFERENT (they changed, or see changed tokens)


## Next Steps

The architecture is complete. To get real text generation:
1. **Phase 7**: Data loading (tinyshakespeare, sliding window)
2. **Phase 8**: Training loop (cross-entropy loss, optimizer, generate samples)
3. Or: load pretrained GPT-2 weights from OpenAI (Chapter 5.6 of the book)